# Phase 1: Passive Tracer Verification

Verifies that passive tracers are correctly advected by the MPAS dynamical core.
Checks mass conservation (should hold to machine precision with the monotonic limiter)
and spatial distribution of tracers initialized as a Gaussian blob.

**Pre-requisite:** Run the Phase 1 container test (see `verification/README.md`).
Expects `data/jw_480km_tracers/output.nc`.

In [ ]:
import netCDF4 as nc
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path("..") / "data" / "jw_480km_tracers"
OUTPUT = DATA_DIR / "output.nc"
assert OUTPUT.exists(), f"Run the Phase 1 test first — {OUTPUT} not found"

ds = nc.Dataset(OUTPUT)
lat = np.degrees(ds["latCell"][:])
lon = np.degrees(ds["lonCell"][:])
nTimes = ds.dimensions["Time"].size
print(f"Grid: {ds.dimensions['nCells'].size} cells, {nTimes} time steps")

## 1. Tracer Mass Conservation

Total tracer mass over time. With monotonic transport, mass should be conserved
to machine precision.

In [ ]:
# TODO: Update variable names once Phase 1 tracers are defined in Registry.xml
tracer_vars = [v for v in ds.variables if v.startswith("tracer")]
if not tracer_vars:
    print("No tracer variables found — Phase 1 not yet implemented")
else:
    fig, ax = plt.subplots(figsize=(10, 5))
    for tv in tracer_vars:
        mass = np.array([ds[tv][t, :, :].sum() for t in range(nTimes)])
        rel_change = (mass - mass[0]) / mass[0]
        ax.plot(range(nTimes), rel_change, "o-", label=tv, markersize=3)
    ax.set_xlabel("Time step")
    ax.set_ylabel("Relative mass change")
    ax.set_title("Tracer Mass Conservation")
    ax.legend()
    ax.axhline(0, color="k", linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.show()

## 2. Tracer Spatial Distribution

Map of tracer concentration at the lowest level, initial and final time.
The Gaussian blob should advect with the flow while preserving its shape.

In [ ]:
if tracer_vars:
    tv = tracer_vars[0]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, t, label in zip(axes, [0, -1], ["Initial", "Final"]):
        vals = ds[tv][t, :, 0]  # lowest level
        sc = ax.scatter(lon, lat, c=vals, s=4, cmap="YlOrRd")
        ax.set_xlabel("Longitude (°)")
        ax.set_ylabel("Latitude (°)")
        ax.set_title(f"{tv} at lowest level — {label}")
        plt.colorbar(sc, ax=ax)
    plt.tight_layout()
    plt.show()
else:
    print("Skipping — no tracer variables")

## 3. Negative Value Check

With the monotonic limiter, tracers should remain non-negative.

In [ ]:
if tracer_vars:
    print("Negative value check:")
    all_pass = True
    for tv in tracer_vars:
        min_val = ds[tv][:].min()
        ok = min_val >= 0
        if not ok: all_pass = False
        print(f"  [{('PASS' if ok else 'FAIL')}] {tv}: min = {min_val:.3e}")
    print(f"\n  Phase 1: {'PASSED' if all_pass else 'FAILED'}")
else:
    print("Phase 1 not yet implemented")

ds.close()